# Light-induced exciton torque from the coherent saddle

This notebook computes the terms generated by a classical light source coupled directly to the exciton field. The goal is to keep the algebra parallel to the `StaticBethe` and `StaticSpinSucep` notebooks, but now starting from the coherent saddle contribution

$$S_{\rm light}[M,F]= -\bar F\circ D_M\circ F.$$

All code comments are in English. The discussion cells spell out which object is being calculated and which approximations are being used.


## Saddle contribution

For the exciton field with external source,

$$S_\Phi=\bar\Phi\circ D_M^{-1}\circ\Phi-\bar\Phi\circ F-\bar F\circ\Phi,$$

the saddle satisfies

$$D_M^{-1}\circ\Phi_0=F.$$

Using the contour object directly, the coherent contribution is

$$S_{\rm light}[M,F]=-\bar F\circ D_M\circ F.$$

We keep this form unsymmetrized. The real/hermitian or retarded projection is a later step, after applying the appropriate Langreth rules to the final kernel.


## Magnetization derivatives

The magnetization dependence enters through

$$D_M^{-1}=V^{-1}-\Pi_M.$$

Therefore

$$\delta D_M=D_M\circ\delta\Pi_M\circ D_M.$$

With

$$\Lambda_l^a=\frac{\delta\Pi_M}{\delta M_l^a},\qquad
\Lambda_{ll'}^{ab}=\frac{\delta^2\Pi_M}{\delta M_l^a\delta M_{l'}^b},$$

the first derivative gives the coherent light-induced field

$$B_{l,a}^{\rm light}\sim -\bar F\circ D\circ\Lambda_l^a\circ D\circ F,$$

and the second derivative gives two contributions:

$$R_{ll',ab}^{I,{\rm light}}
= -\bar F\circ\left[D\Lambda_l^aD\Lambda_{l'}^bD+D\Lambda_{l'}^bD\Lambda_l^aD\right]\circ F,$$

$$R_{ll',ab}^{II,{\rm light}}
= -\bar F\circ D\Lambda_{ll'}^{ab}D\circ F.$$

The vertices are the same derivatives of the electron-hole bubble used in the static spin-susceptibility notebook. What changes is the contraction: the exciton-channel trace is replaced by external source legs $\bar F$ and $F$.


## What this numerical version computes

There are two frequency roles that should not be mixed:

- $\omega_L$: the frequency selected by the light source.
- $\Omega$: the frequency conjugate to the relative time of the response kernel $R(z,z')$.

The source frequency is applied analytically. For a complex monochromatic source, $F(\nu)$ selects $\nu=\omega_L$; for a real harmonic source, it selects $\nu=\pm\omega_L$. No numerical delta is needed in the theory.

The existing code below is a first static benchmark: it evaluates the zero-response-frequency limit, $\Omega=0$, using the same one-frequency Lehmann vertices as the previous notebooks. The full frequency-dependent light-induced kernel $R(\Omega)$ requires the multifrequency vertices shown in the analytic formulas below.

## Static limit currently evaluated by the code

The retarded exciton propagator is

$$D^R(\nu)=\left[V^{-1}\mathbf{1}-\Pi^R(\nu)\right]^{-1},$$

where $\nu$ is the exciton/source frequency running through the light line. In the static response limit, $\Omega=0$, the vertices do not transfer frequency, so

$$\Lambda_l^{a,R}(\nu,\nu)\equiv\Lambda_l^{a,R}(\nu),\qquad
\Lambda_{ll'}^{ab,R}(\nu,\nu;0)\equiv\Lambda_{ll'}^{ab,R}(\nu).$$

Then the coherent saddle contribution used by the current code is

$$S_{\mathrm{light}}[M,F]=-
\int\frac{d\nu}{2\pi}\,
F^\dagger(\nu)D_M^R(\nu)F(\nu).$$

For a complex monochromatic source,

$$F_\mu(\nu)=2\pi f_\mu\delta(\nu-\omega_L),$$

this becomes, per unit observation time,

$$\frac{S_{\mathrm{light}}}{T_{\mathrm{obs}}}=-f^\dagger D_M^R(\omega_L)f.$$

The same analytic replacement is used for the static field and curvature: evaluate the one-frequency expressions at $\nu=\omega_L$ for complex light, or sum the two contributions $\nu=\pm\omega_L$ for a real harmonic field.

In [ ]:
using LinearAlgebra
using Printf
using PyPlot

rc("font", family="serif")
rc("font", size=13)
rc("axes", labelsize=14, titlesize=14)
rc("legend", fontsize=10);


In [ ]:
# -----------------------------
# Physical and numerical parameters
# -----------------------------
L = 80
γ = 1.0
Δ_layer = 8.0
Jsd = 0.35
Mmag = 1.0
θ = pi/4
V = -1.2
T = 0.08

# Chemical potentials diagonal in the Hamiltonian.
μchem = 0.0
μbias = 0.0
μ1 = μchem + μbias/2
μ2 = μchem - μbias/2

# Symmetric frequency grid for the drive integral.
ηΩ = 0.25
Ωmin, Ωmax, NΩ = -15.0, 15.0, 401

ks = collect(range(-π, stop=π - 2π/L, length=L))
Ωs = collect(range(Ωmin, Ωmax, length=NΩ));


In [ ]:
# Pauli matrices in physical spin space.
σ0 = ComplexF64[1 0; 0 1]
σx = ComplexF64[0 1; 1 0]
σy = ComplexF64[0 -1im; 1im 0]
σz = ComplexF64[1 0; 0 -1]

# Physical spin vertices s^a = sigma^a/2.
S_spin = [σx, σy, σz] ./ 2
spin_labels = ["x", "y", "z"]

# Excitonic vertices Gamma^mu = sigma^mu/sqrt(2), mu=0,x,y,z.
Γ_exciton = [σ0, σx, σy, σz] ./ sqrt(2)
channel_labels = ["0", "x", "y", "z"]

function dh_dM(layer::Int, comp::Int)
    # This derivative follows the current h_layer convention, including the layer-dependent sign.
    layer_sign = (-1)^(layer + 1)
    return layer_sign * (-Jsd) * (2 * S_spin[comp])
end;


In [ ]:
ϵk(k; γ=γ) = -2γ*cos(k)

function fermi(E; T=T, μ=0.0)
    x = (E - μ) / T
    x > 60 && return 0.0
    x < -60 && return 1.0
    return 1 / (exp(x) + 1)
end

layer_chemical_potential(layer::Int) = layer == 1 ? μ1 : μ2

function h_layer(k, M; layer::Int)
    offset = layer == 1 ? +Δ_layer/2 : Δ_layer/2
    energy = (ϵk(k) + offset) * σ0 - Jsd * (M[1]*σx + M[2]*σy + M[3]*σz)
    energy .*= (-1)^(layer+1)
    energy .-= layer_chemical_potential(layer) * σ0
    return energy
end

function magnetizations(θ)
    M1 = Mmag .* [0.0, 0.0, 1.0]
    M2 = Mmag .* [sin(θ), 0.0, cos(θ)]
    return M1, M2
end

function diagonalize_layers(ks, θ)
    M1, M2 = magnetizations(θ)
    E1 = zeros(Float64, 2, length(ks))
    E2 = zeros(Float64, 2, length(ks))
    U1 = Vector{Matrix{ComplexF64}}(undef, length(ks))
    U2 = Vector{Matrix{ComplexF64}}(undef, length(ks))

    for (ik, k) in enumerate(ks)
        F1 = eigen(Hermitian(h_layer(k, M1; layer=1)))
        F2 = eigen(Hermitian(h_layer(k, M2; layer=2)))
        E1[:, ik] .= F1.values
        E2[:, ik] .= F2.values
        U1[ik] = Matrix{ComplexF64}(F1.vectors)
        U2[ik] = Matrix{ComplexF64}(F2.vectors)
    end

    return (; M1, M2, E1, E2, U1, U2)
end;


## Exciton BSE block

This is the same equilibrium BSE block used before. It computes the electron-hole bubble $\Pi^R_{\mu\nu}(\Omega)$ and the dressed exciton propagator

$$D^R(\Omega)=\left[V^{-1}\mathbf{1}-\Pi^R(\Omega)\right]^{-1}.$$


In [ ]:
function diagonalize_layers_from_M(ks, M1, M2)
    E1 = zeros(Float64, 2, length(ks))
    E2 = zeros(Float64, 2, length(ks))
    U1 = Vector{Matrix{ComplexF64}}(undef, length(ks))
    U2 = Vector{Matrix{ComplexF64}}(undef, length(ks))

    for (ik, k) in enumerate(ks)
        F1 = eigen(Hermitian(h_layer(k, M1; layer=1)))
        F2 = eigen(Hermitian(h_layer(k, M2; layer=2)))
        E1[:, ik] .= F1.values
        E2[:, ik] .= F2.values
        U1[ik] = Matrix{ComplexF64}(F1.vectors)
        U2[ik] = Matrix{ComplexF64}(F2.vectors)
    end

    return (; M1, M2, E1, E2, U1, U2)
end

function polarization_R_band_from_M(Ωs, ks, M1, M2; η=ηΩ)
    data = diagonalize_layers_from_M(ks, M1, M2)
    Π = zeros(ComplexF64, 4, 4, length(Ωs))

    for (ik, _) in enumerate(ks)
        U1k = data.U1[ik]
        U2k = data.U2[ik]
        Mμ = [U1k' * Γ_exciton[μ] * U2k for μ in 1:4]

        for a in 1:2, b in 1:2
            ΔE = data.E1[a, ik] - data.E2[b, ik]
            occ = fermi(data.E2[b, ik]) - fermi(data.E1[a, ik])

            for (iΩ, Ω) in enumerate(Ωs)
                pref = occ / (Ω + 1im*η - ΔE) / length(ks)
                for μ in 1:4, ν in 1:4
                    Π[μ, ν, iΩ] += pref * Mμ[μ][a, b] * conj(Mμ[ν][a, b])
                end
            end
        end
    end

    return Π, data
end

function exciton_propagator(Π; V=V)
    D = similar(Π)
    VinvI = (1/V) * Matrix{ComplexF64}(I, 4, 4)
    for iΩ in axes(Π, 3)
        D[:, :, iΩ] .= inv(VinvI - Π[:, :, iΩ])
    end
    return D
end;


function precompute_exciton_D(Ωs, ks, θ; V=V, η=ηΩ)
    M1, M2 = magnetizations(θ)
    Π, data = polarization_R_band_from_M(Ωs, ks, M1, M2; η=η)
    D = exciton_propagator(Π; V=V)
    return D, Π, data
end;


## Band/Lehmann vertices

The light-induced curvature uses the same analytic derivatives of $\Pi_M$ as the previous spin-susceptibility notebook:

$$\Lambda_l^a=\delta_{M_l^a}\Pi_M,$$

$$\Lambda_{ll'}^{ab}=\delta_{M_l^a}\delta_{M_{l'}^b}\Pi_M.$$

The code below evaluates them in the eigenbasis of the frozen layer Hamiltonians. This avoids finite differences in $M$.


## Lehmann representation of the retarded blocks

The fast band method evaluates the retarded component directly. For each momentum, diagonalize the frozen layer Hamiltonians,

$$h_l(k)|u^l_{n k}\rangle=E^l_{n k}|u^l_{n k}\rangle,$$

and define matrix elements in the band basis

$$\Gamma^{\mu}_{pq}(k)=\langle u^1_{p k}|\Gamma_\mu|u^2_{q k}\rangle,$$

$$h^{l,a}_{mn}(k)=\langle u^l_{m k}|\partial_{M_l^a}h_l|u^l_{n k}\rangle.$$

The retarded electron-hole kernel used throughout the notebook is

$$K^R_\eta(\Omega;x,y)=\frac{f(y)-f(x)}{\Omega+i\eta-x+y}.$$

Thus the bubble is

$$\Pi^R_{\mu\nu}(\Omega)=\frac{1}{L}\sum_{k,p,q}
K^R_\eta(\Omega;E^1_{p k},E^2_{q k})
\Gamma^\mu_{pq}(k)\Gamma^{\nu *}_{pq}(k).$$

The one-magnetization vertices are divided differences of the same retarded kernel. For a derivative on layer 1,

$$\Lambda^{1,a,R}_{\mu\nu}(\Omega)=\frac{1}{L}\sum_{k,p,r,q}
K_x^R(\Omega;E^1_{p k},E^1_{r k};E^2_{q k})
h^{1,a}_{pr}(k)\Gamma^\mu_{rq}(k)\Gamma^{\nu *}_{pq}(k),$$

and for a derivative on layer 2,

$$\Lambda^{2,a,R}_{\mu\nu}(\Omega)=\frac{1}{L}\sum_{k,p,q,r}
K_y^R(\Omega;E^1_{p k};E^2_{q k},E^2_{r k})
\Gamma^\mu_{pq}(k)h^{2,a}_{qr}(k)\Gamma^{\nu *}_{pr}(k).$$

Here $K_x^R$ and $K_y^R$ are first divided differences of $K^R$ with respect to the first-layer and second-layer energies. When two nodes coincide, the code replaces the divided difference by the analytic derivative.

The two-magnetization vertex uses second divided differences. For the layer pair $(1,1)$,

$$\Lambda^{11,ab,R}_{\mu\nu}=\frac{1}{L}\sum_{k,p,r,s,q}
K_{xx}^R(\Omega;E^1_p,E^1_r,E^1_s;E^2_q)
\left[h^{1,b}_{pr}h^{1,a}_{rs}+h^{1,a}_{pr}h^{1,b}_{rs}\right]
\Gamma^\mu_{sq}\Gamma^{\nu *}_{pq}.$$

For $(2,2)$,

$$\Lambda^{22,ab,R}_{\mu\nu}=\frac{1}{L}\sum_{k,p,q,r,s}
K_{yy}^R(\Omega;E^1_p;E^2_q,E^2_r,E^2_s)
\Gamma^\mu_{pq}
\left[h^{2,a}_{qr}h^{2,b}_{rs}+h^{2,b}_{qr}h^{2,a}_{rs}\right]
\Gamma^{\nu *}_{ps}.$$

For the mixed layer pair $(1,2)$,

$$\Lambda^{12,ab,R}_{\mu\nu}=\frac{1}{L}\sum_{k,p,r,q,s}
K_{xy}^R(\Omega;E^1_p,E^1_r;E^2_q,E^2_s)
h^{1,a}_{pr}\Gamma^\mu_{rq}h^{2,b}_{qs}\Gamma^{\nu *}_{ps},$$

and $(2,1)$ is obtained by swapping the external derivatives,

$$\Lambda^{21,ab,R}_{\mu\nu}=\frac{1}{L}\sum_{k,p,r,q,s}
K_{xy}^R(\Omega;E^1_p,E^1_r;E^2_q,E^2_s)
h^{1,b}_{pr}\Gamma^\mu_{rq}h^{2,a}_{qs}\Gamma^{\nu *}_{ps}.$$

The functions `kernel_K`, `dd1_x`, `dd1_y`, `dd2_x`, `dd2_y`, and `dd_xy` implement these retarded kernels and divided differences. The $+i\eta$ prescription is what fixes the retarded component.


In [ ]:
function fermi_prime(E; T=T, μ=0.0)
    f = fermi(E; T=T, μ=μ)
    return -(f * (1 - f)) / T
end

function fermi_second(E; T=T, μ=0.0)
    f = fermi(E; T=T, μ=μ)
    return f * (1 - f) * (1 - 2f) / T^2
end

function kernel_K(Ω, x, y, η=ηΩ)
    q = Ω + 1im*η - x + y
    return (fermi(y) - fermi(x)) / q
end

function kernel_Kx(Ω, x, y, η=ηΩ)
    q = Ω + 1im*η - x + y
    n = fermi(y) - fermi(x)
    return -fermi_prime(x) / q + n / q^2
end

function kernel_Ky(Ω, x, y, η=ηΩ)
    q = Ω + 1im*η - x + y
    n = fermi(y) - fermi(x)
    return fermi_prime(y) / q - n / q^2
end

function kernel_Kxx(Ω, x, y, η=ηΩ)
    q = Ω + 1im*η - x + y
    n = fermi(y) - fermi(x)
    return -fermi_second(x) / q - 2 * fermi_prime(x) / q^2 + 2 * n / q^3
end

function kernel_Kyy(Ω, x, y, η=ηΩ)
    q = Ω + 1im*η - x + y
    n = fermi(y) - fermi(x)
    return fermi_second(y) / q - 2 * fermi_prime(y) / q^2 + 2 * n / q^3
end

function kernel_Kxy(Ω, x, y, η=ηΩ)
    q = Ω + 1im*η - x + y
    n = fermi(y) - fermi(x)
    return (fermi_prime(x) + fermi_prime(y)) / q^2 - 2 * n / q^3
end

same_node(a, b) = abs(a - b) < 1e-10

function dd1_x(Ω, x1, x2, y, η=ηΩ)
    same_node(x1, x2) && return kernel_Kx(Ω, x1, y, η)
    return (kernel_K(Ω, x1, y, η) - kernel_K(Ω, x2, y, η)) / (x1 - x2)
end

function dd1_y(Ω, x, y1, y2, η=ηΩ)
    same_node(y1, y2) && return kernel_Ky(Ω, x, y1, η)
    return (kernel_K(Ω, x, y1, η) - kernel_K(Ω, x, y2, η)) / (y1 - y2)
end

function dd2_x(Ω, x1, x2, x3, y, η=ηΩ)
    if same_node(x1, x2) && same_node(x2, x3)
        return kernel_Kxx(Ω, x1, y, η) / 2
    elseif same_node(x1, x2)
        return (kernel_Kx(Ω, x1, y, η) - dd1_x(Ω, x1, x3, y, η)) / (x1 - x3)
    elseif same_node(x2, x3)
        return (dd1_x(Ω, x1, x2, y, η) - kernel_Kx(Ω, x2, y, η)) / (x1 - x2)
    elseif same_node(x1, x3)
        return (kernel_Kx(Ω, x1, y, η) - dd1_x(Ω, x1, x2, y, η)) / (x1 - x2)
    else
        return (dd1_x(Ω, x1, x2, y, η) - dd1_x(Ω, x2, x3, y, η)) / (x1 - x3)
    end
end

function dd2_y(Ω, x, y1, y2, y3, η=ηΩ)
    if same_node(y1, y2) && same_node(y2, y3)
        return kernel_Kyy(Ω, x, y1, η) / 2
    elseif same_node(y1, y2)
        return (kernel_Ky(Ω, x, y1, η) - dd1_y(Ω, x, y1, y3, η)) / (y1 - y3)
    elseif same_node(y2, y3)
        return (dd1_y(Ω, x, y1, y2, η) - kernel_Ky(Ω, x, y2, η)) / (y1 - y2)
    elseif same_node(y1, y3)
        return (kernel_Ky(Ω, x, y1, η) - dd1_y(Ω, x, y1, y2, η)) / (y1 - y2)
    else
        return (dd1_y(Ω, x, y1, y2, η) - dd1_y(Ω, x, y2, y3, η)) / (y1 - y3)
    end
end

function dd_xy(Ω, x1, x2, y1, y2, η=ηΩ)
    if same_node(x1, x2) && same_node(y1, y2)
        return kernel_Kxy(Ω, x1, y1, η)
    elseif same_node(x1, x2)
        return (kernel_Kx(Ω, x1, y1, η) - kernel_Kx(Ω, x1, y2, η)) / (y1 - y2)
    elseif same_node(y1, y2)
        return (kernel_Ky(Ω, x1, y1, η) - kernel_Ky(Ω, x2, y1, η)) / (x1 - x2)
    else
        return (kernel_K(Ω, x1, y1, η) - kernel_K(Ω, x2, y1, η) -
                kernel_K(Ω, x1, y2, η) + kernel_K(Ω, x2, y2, η)) / ((x1 - x2) * (y1 - y2))
    end
end

function projectors_from_eigenvectors(U)
    return [U[:, n] * U[:, n]' for n in 1:2]
end

function lambda_layerpair_diag_band(Ωs, ks, θ; layer_pair=(1, 1), η=ηΩ)
    l, lp = layer_pair
    @assert l in (1, 2) "l must be 1 or 2"
    @assert lp in (1, 2) "lp must be 1 or 2"

    data = diagonalize_layers(ks, θ)
    Λdiag = zeros(ComplexF64, 3, 4, 4, length(Ωs))
    dh1 = [dh_dM(1, a) for a in 1:3]
    dh2 = [dh_dM(2, a) for a in 1:3]

    for (ik, _) in enumerate(ks)
        E1 = data.E1[:, ik]
        E2 = data.E2[:, ik]
        U1 = data.U1[ik]
        U2 = data.U2[ik]
        dh1e = [U1' * dh1[a] * U1 for a in 1:3]
        dh2e = [U2' * dh2[a] * U2 for a in 1:3]
        Γe = [U1' * Γ_exciton[μ] * U2 for μ in 1:4]

        for (iΩ, Ω) in enumerate(Ωs)
            for a in 1:3, μ in 1:4, ν in 1:4
                acc = zero(ComplexF64)
                Γμ = Γe[μ]
                Γν = Γe[ν]
                h1a = dh1e[a]
                h2a = dh2e[a]

                if l == 1 && lp == 1
                    for p in 1:2, r in 1:2, s in 1:2, q in 1:2
                        coeff = 2 * dd2_x(Ω, E1[p], E1[r], E1[s], E2[q], η)
                        acc += coeff * h1a[p, r] * h1a[r, s] * Γμ[s, q] * conj(Γν[p, q])
                    end
                elseif l == 2 && lp == 2
                    for p in 1:2, q in 1:2, r in 1:2, s in 1:2
                        coeff = 2 * dd2_y(Ω, E1[p], E2[q], E2[r], E2[s], η)
                        acc += coeff * Γμ[p, q] * h2a[q, r] * h2a[r, s] * conj(Γν[p, s])
                    end
                else
                    for p in 1:2, r in 1:2, q in 1:2, s in 1:2
                        coeff = dd_xy(Ω, E1[p], E1[r], E2[q], E2[s], η)
                        acc += coeff * h1a[p, r] * Γμ[r, q] * h2a[q, s] * conj(Γν[p, s])
                    end
                end

                Λdiag[a, μ, ν, iΩ] += acc / length(ks)
            end
        end
    end

    return Λdiag, data
end

function spin_susceptibility_oneD_diag_band(D, Ωs, ks, θ; layer_pair=(1, 1), η=ηΩ, prefactor=-1im)
    Λdiag, data = lambda_layerpair_diag_band(Ωs, ks, θ; layer_pair=layer_pair, η=η)
    Rdiag = contract_singleD_diag(D, Λdiag; prefactor=prefactor)
    return Rdiag, Λdiag, data
end

function spin_susceptibility_oneD_diag_band(Ωs, ks, θ; layer_pair=(1, 1), η=ηΩ, prefactor=-1im, V=V)
    D, Π, dataD = precompute_exciton_D(Ωs, ks, θ; V=V, η=η)
    Rdiag, Λdiag, dataΛ = spin_susceptibility_oneD_diag_band(D, Ωs, ks, θ; layer_pair=layer_pair, η=η, prefactor=prefactor)
    return Rdiag, Λdiag, D, Π, dataD, dataΛ
end;

function lambda_layerpair_full_band(Ωs, ks, θ; layer_pair=(1, 1), η=ηΩ)
    l, lp = layer_pair
    @assert l in (1, 2) "l must be 1 or 2"
    @assert lp in (1, 2) "lp must be 1 or 2"

    data = diagonalize_layers(ks, θ)
    Λ = zeros(ComplexF64, 3, 3, 4, 4, length(Ωs))
    dh1 = [dh_dM(1, a) for a in 1:3]
    dh2 = [dh_dM(2, a) for a in 1:3]

    for (ik, _) in enumerate(ks)
        E1 = data.E1[:, ik]
        E2 = data.E2[:, ik]
        U1 = data.U1[ik]
        U2 = data.U2[ik]
        dh1e = [U1' * dh1[a] * U1 for a in 1:3]
        dh2e = [U2' * dh2[a] * U2 for a in 1:3]
        Γe = [U1' * Γ_exciton[μ] * U2 for μ in 1:4]

        for (iΩ, Ω) in enumerate(Ωs)
            for a in 1:3, b in 1:3, μ in 1:4, ν in 1:4
                acc = zero(ComplexF64)
                Γμ = Γe[μ]
                Γν = Γe[ν]
                h1a = dh1e[a]
                h1b = dh1e[b]
                h2a = dh2e[a]
                h2b = dh2e[b]

                if l == 1 && lp == 1
                    for p in 1:2, r in 1:2, s in 1:2, q in 1:2
                        coeff = dd2_x(Ω, E1[p], E1[r], E1[s], E2[q], η)
                        acc += coeff * (h1b[p, r] * h1a[r, s] + h1a[p, r] * h1b[r, s]) * Γμ[s, q] * conj(Γν[p, q])
                    end
                elseif l == 2 && lp == 2
                    for p in 1:2, q in 1:2, r in 1:2, s in 1:2
                        coeff = dd2_y(Ω, E1[p], E2[q], E2[r], E2[s], η)
                        acc += coeff * Γμ[p, q] * (h2a[q, r] * h2b[r, s] + h2b[q, r] * h2a[r, s]) * conj(Γν[p, s])
                    end
                elseif l == 1 && lp == 2
                    for p in 1:2, r in 1:2, q in 1:2, s in 1:2
                        coeff = dd_xy(Ω, E1[p], E1[r], E2[q], E2[s], η)
                        acc += coeff * h1a[p, r] * Γμ[r, q] * h2b[q, s] * conj(Γν[p, s])
                    end
                else
                    for p in 1:2, r in 1:2, q in 1:2, s in 1:2
                        coeff = dd_xy(Ω, E1[p], E1[r], E2[q], E2[s], η)
                        acc += coeff * h1b[p, r] * Γμ[r, q] * h2a[q, s] * conj(Γν[p, s])
                    end
                end

                Λ[a, b, μ, ν, iΩ] += acc / length(ks)
            end
        end
    end

    return Λ, data
end

function contract_singleD_full(D, Λ; prefactor=-1im)
    R = zeros(ComplexF64, 3, 3, size(D, 3))

    for a in 1:3, b in 1:3, iΩ in axes(D, 3)
        acc = zero(ComplexF64)
        for μ in 1:4, ν in 1:4
            acc += D[μ, ν, iΩ] * Λ[a, b, ν, μ, iΩ]
        end
        R[a, b, iΩ] = prefactor * acc
    end

    return R
end

function spin_susceptibility_oneD_full_band(D, Ωs, ks, θ; layer_pair=(1, 1), η=ηΩ, prefactor=-1im)
    Λ, data = lambda_layerpair_full_band(Ωs, ks, θ; layer_pair=layer_pair, η=η)
    R = contract_singleD_full(D, Λ; prefactor=prefactor)
    return R, Λ, data
end;


## One-magnetization vertex

The two-single-vertex light term requires $\Lambda_l^a$ for both layers. This is the one-derivative vertex already used in the two-$D$ spin-susceptibility correction.


In [ ]:
function lambda_one_spin_layers_band(Ωs, ks, θ; η=ηΩ)
    data = diagonalize_layers(ks, θ)
    Λone = zeros(ComplexF64, 2, 3, 4, 4, length(Ωs))
    dh1 = [dh_dM(1, a) for a in 1:3]
    dh2 = [dh_dM(2, a) for a in 1:3]

    for (ik, _) in enumerate(ks)
        E1 = data.E1[:, ik]
        E2 = data.E2[:, ik]
        U1 = data.U1[ik]
        U2 = data.U2[ik]
        dh1e = [U1' * dh1[a] * U1 for a in 1:3]
        dh2e = [U2' * dh2[a] * U2 for a in 1:3]
        Γe = [U1' * Γ_exciton[μ] * U2 for μ in 1:4]

        for (iΩ, Ω) in enumerate(Ωs)
            for a in 1:3, μ in 1:4, ν in 1:4
                Γμ = Γe[μ]
                Γν = Γe[ν]
                h1a = dh1e[a]
                h2a = dh2e[a]

                acc1 = zero(ComplexF64)
                for p in 1:2, r in 1:2, q in 1:2
                    coeff = dd1_x(Ω, E1[p], E1[r], E2[q], η)
                    acc1 += coeff * h1a[p, r] * Γμ[r, q] * conj(Γν[p, q])
                end

                acc2 = zero(ComplexF64)
                for p in 1:2, q in 1:2, r in 1:2
                    coeff = dd1_y(Ω, E1[p], E2[q], E2[r], η)
                    acc2 += coeff * Γμ[p, q] * h2a[q, r] * conj(Γν[p, r])
                end

                Λone[1, a, μ, ν, iΩ] += acc1 / length(ks)
                Λone[2, a, μ, ν, iΩ] += acc2 / length(ks)
            end
        end
    end

    return Λone, data
end

function contract_twoD(D, Λone; prefactor=1im)
    R2D = zeros(ComplexF64, 2, 2, 3, 3, size(D, 3))

    @views for iΩ in axes(D, 3), l in 1:2, lp in 1:2, a in 1:3, b in 1:3
        R2D[l, lp, a, b, iΩ] = prefactor * tr(D[:, :, iΩ] * Λone[l, a, :, :, iΩ] * D[:, :, iΩ] * Λone[lp, b, :, :, iΩ])
    end

    return R2D
end

function spin_susceptibility_twoD_band(D, Ωs, ks, θ; η=ηΩ, prefactor=1im)
    Λone, data = lambda_one_spin_layers_band(Ωs, ks, θ; η=η)
    R2D = contract_twoD(D, Λone; prefactor=prefactor)
    return R2D, Λone, data
end

function spin_susceptibility_twoD_diag_band(D, Ωs, ks, θ; layer_pair=(1, 1), η=ηΩ, prefactor=1im)
    R2D, Λone, data = spin_susceptibility_twoD_band(D, Ωs, ks, θ; η=η, prefactor=prefactor)
    l, lp = layer_pair
    Rdiag = zeros(ComplexF64, 3, length(Ωs))
    for a in 1:3, iΩ in axes(Ωs, 1)
        Rdiag[a, iΩ] = R2D[l, lp, a, a, iΩ]
    end
    return Rdiag, R2D, Λone, data
end;


## Light source

The source lives in the exciton-channel space $\mu=0,x,y,z$. The default choice follows the PDF discussion: non-polarized light in the excitonic charge channel,

$$F^\mu(\nu)=(F^0(\nu),0,0,0).$$

For a monochromatic calculation the delta function is applied analytically:

$$F_\mu(\nu)=2\pi f_\mu\delta(\nu-\omega_L).$$

For a real harmonic field there are two analytic components at $\nu=\pm\omega_L$. For a Gaussian pulse, $F(\nu)$ is a smooth spectrum and the remaining $d\nu/(2\pi)$ integral is performed numerically.

In [ ]:
# -----------------------------
# Coherent light source in the exciton-channel space
# -----------------------------
nearest_frequency_index(Ωs, Ω0) = argmin(abs.(Ωs .- Ω0))

function channel_vector(channel, coefficient)
    vec = zeros(ComplexF64, 4)
    vec[channel] = coefficient
    return vec
end

function monochromatic_drive_components(; channel=1, amplitude=1.0, phase=0.0,
                                        ωL=5.0, real_drive=true)
    components = Any[]

    if real_drive
        # Convention: F(t)=A cos(ωL t + phase).
        push!(components, (; label="+omega_L", frequency=ωL,
                           vector=channel_vector(channel, 0.5 * amplitude * cis(-phase))))
        push!(components, (; label="-omega_L", frequency=-ωL,
                           vector=channel_vector(channel, 0.5 * amplitude * cis( phase))))
        kind = :monochromatic_real
    else
        # Convention: F(t)=A exp(-i ωL t + i phase).
        push!(components, (; label="+omega_L", frequency=ωL,
                           vector=channel_vector(channel, amplitude * cis(phase))))
        kind = :monochromatic_complex
    end

    info = (; kind=kind, channel=channel, amplitude=amplitude, phase=phase,
            ωL=ωL, components=components, use_frequency_integral=false,
            normalization=:analytic_delta)
    return info
end

function marker_spectrum_for_components(Ωs, components)
    # This array is only used to mark analytic source frequencies in plots.
    FΩs = zeros(ComplexF64, 4, length(Ωs))
    for comp in components
        idx = nearest_frequency_index(Ωs, comp.frequency)
        FΩs[:, idx] .+= comp.vector
    end
    return FΩs
end

function gaussian_pulse_spectrum(Ωs; channel=1, amplitude=1.0, ωL=5.0, σt=8.0,
                                 t0=30.0, phase=0.0, normalize_peak=true)
    # Forward convention: F(ν) = ∫ dt exp(i ν t) F(t).
    # F(t) = A exp(-(t-t0)^2/(2 σt^2)) cos(ωL t + phase).
    vals = ComplexF64[]
    pref = sqrt(2π) * σt / 2
    for ν in Ωs
        pos = exp(1im * (ν + ωL) * t0 + 1im * phase) * exp(-0.5 * σt^2 * (ν + ωL)^2)
        neg = exp(1im * (ν - ωL) * t0 - 1im * phase) * exp(-0.5 * σt^2 * (ν - ωL)^2)
        push!(vals, pref * (pos + neg))
    end

    if normalize_peak
        peak = maximum(abs.(vals))
        peak > 0 && (vals .= amplitude .* vals ./ peak)
    else
        vals .*= amplitude
    end

    FΩs = zeros(ComplexF64, 4, length(Ωs))
    FΩs[channel, :] .= vals
    info = (; kind=:gaussian, channel=channel, amplitude=amplitude, phase=phase,
            ωL=ωL, σt=σt, t0=t0, normalize_peak=normalize_peak,
            use_frequency_integral=true)
    return FΩs, info
end

function flat_debug_drive_spectrum(Ωs; channel=1, amplitude=1.0, phase=0.0)
    FΩs = zeros(ComplexF64, 4, length(Ωs))
    FΩs[channel, :] .= amplitude * cis(phase)
    info = (; kind=:flat_debug, channel=channel, amplitude=amplitude, phase=phase,
            use_frequency_integral=true)
    return FΩs, info
end

function make_drive_spectrum_with_info(Ωs; envelope=:monochromatic_real, channel=1,
                                       amplitude=1.0, phase=0.0, ωL=5.0,
                                       σt=8.0, t0=30.0, normalize_peak=true)
    if envelope == :monochromatic_real
        info = monochromatic_drive_components(; channel=channel, amplitude=amplitude,
                                              phase=phase, ωL=ωL, real_drive=true)
        return marker_spectrum_for_components(Ωs, info.components), info
    elseif envelope == :monochromatic_complex
        info = monochromatic_drive_components(; channel=channel, amplitude=amplitude,
                                              phase=phase, ωL=ωL, real_drive=false)
        return marker_spectrum_for_components(Ωs, info.components), info
    elseif envelope == :gaussian
        return gaussian_pulse_spectrum(Ωs; channel=channel, amplitude=amplitude,
                                       ωL=ωL, σt=σt, t0=t0, phase=phase,
                                       normalize_peak=normalize_peak)
    elseif envelope == :flat_debug
        return flat_debug_drive_spectrum(Ωs; channel=channel, amplitude=amplitude, phase=phase)
    end
    error("Unknown drive envelope: " * string(envelope))
end

function make_drive_spectrum(Ωs; kwargs...)
    FΩs, _ = make_drive_spectrum_with_info(Ωs; kwargs...)
    return FΩs
end;

## Light-induced contractions

The variables $z,z'$ in $R(z,z')$ introduce the response frequency $\Omega$, conjugate to the relative time $z-z'$. The frequency carried by the light/exciton line is denoted by $\nu$.

The one-magnetization vertex transfers frequency. We write it as

$$\Lambda_l^{a,R}(\nu+\Omega,\nu),$$

meaning that the exciton line enters the vertex with frequency $\nu$ and leaves with frequency $\nu+\Omega$. The first derivative of the saddle therefore has the analytic structure

$$B_{l,a}(\Omega)=
-\int\frac{d\nu}{2\pi}\,
F^\dagger(\nu+\Omega)D^R(\nu+\Omega)
\Lambda_l^{a,R}(\nu+\Omega,\nu)D^R(\nu)F(\nu).$$

The two-one-vertex contribution to the curvature has two possible orderings of the insertions at $z$ and $z'$:

$$R^{I}_{ll',ab}(\Omega)=
-\int\frac{d\nu}{2\pi}\,
F^\dagger(\nu)D^R(\nu)
\left[
\Lambda_l^{a,R}(\nu,\nu+\Omega)D^R(\nu+\Omega)
\Lambda_{l'}^{b,R}(\nu+\Omega,\nu)
+\Lambda_{l'}^{b,R}(\nu,\nu-\Omega)D^R(\nu-\Omega)
\Lambda_l^{a,R}(\nu-\Omega,\nu)
\right]
D^R(\nu)F(\nu).$$

The two-magnetization vertex keeps the total exciton frequency unchanged but depends on the frequency exchanged between the two insertion points:

$$R^{II}_{ll',ab}(\Omega)=
-\int\frac{d\nu}{2\pi}\,
F^\dagger(\nu)D^R(\nu)
\Lambda_{ll'}^{ab,R}(\nu,\nu;\Omega)D^R(\nu)F(\nu).$$

For complex monochromatic light, the source simply applies the replacement $\nu=\omega_L$ in these expressions. For real harmonic light, one sums the corresponding $\nu=+\omega_L$ and $\nu=-\omega_L$ contributions, with possible fast $2\omega_L$ pieces kept or dropped depending on whether the response is time-averaged.

The code cells below currently evaluate the $\Omega=0$ limit, where

$$\Lambda_l^{a,R}(\nu,\nu)=\Lambda_l^{a,R}(\nu),\qquad
\Lambda_{ll'}^{ab,R}(\nu,\nu;0)=\Lambda_{ll'}^{ab,R}(\nu).$$

Computing the full $R(\Omega)$ requires implementing these multifrequency vertices rather than reusing the one-frequency static vertices.

In [ ]:
# -----------------------------
# Light-induced saddle-point terms in the Ω=0 static limit
# -----------------------------
quadratic_drive(F, A) = dot(F, A * F)  # F(ν)^† A(ν) F(ν)

function light_effective_field_integrand_from_drive(D, Λone, FΩs; prefactor=-1.0)
    @assert size(FΩs, 1) == 4
    @assert size(FΩs, 2) == size(D, 3)
    Bν = zeros(ComplexF64, 2, 3, size(D, 3))

    @views for iν in axes(D, 3), l in 1:2, a in 1:3
        F = FΩs[:, iν]
        Dν = D[:, :, iν]
        Λa = Λone[l, a, :, :, iν]
        Bν[l, a, iν] = prefactor * quadratic_drive(F, Dν * Λa * Dν)
    end

    return Bν
end

function light_kernel_I_integrand_from_drive(D, Λone, FΩs; layer_pair=(1, 2), prefactor=-1.0)
    # Ω=0 limit of the two-one-vertex term.
    @assert size(FΩs, 1) == 4
    @assert size(FΩs, 2) == size(D, 3)
    l, lp = layer_pair
    RIν = zeros(ComplexF64, 3, 3, size(D, 3))

    @views for iν in axes(D, 3), a in 1:3, b in 1:3
        F = FΩs[:, iν]
        Dν = D[:, :, iν]
        Λa = Λone[l, a, :, :, iν]
        Λb = Λone[lp, b, :, :, iν]
        term = Dν * Λa * Dν * Λb * Dν + Dν * Λb * Dν * Λa * Dν
        RIν[a, b, iν] = prefactor * quadratic_drive(F, term)
    end

    return RIν
end

function light_kernel_II_integrand_from_drive(D, Λfull, FΩs; prefactor=-1.0)
    # Ω=0 limit of the two-magnetization-vertex term.
    @assert size(FΩs, 1) == 4
    @assert size(FΩs, 2) == size(D, 3)
    RIIν = zeros(ComplexF64, 3, 3, size(D, 3))

    @views for iν in axes(D, 3), a in 1:3, b in 1:3
        F = FΩs[:, iν]
        Dν = D[:, :, iν]
        Λab = Λfull[a, b, :, :, iν]
        RIIν[a, b, iν] = prefactor * quadratic_drive(F, Dν * Λab * Dν)
    end

    return RIIν
end

function contract_static_light_at_frequency(Dν, Λoneν, Λfullν, F; layer_pair=(1, 2), prefactor=-1.0)
    l, lp = layer_pair
    B = zeros(ComplexF64, 2, 3)
    RI = zeros(ComplexF64, 3, 3)
    RII = zeros(ComplexF64, 3, 3)

    @views for layer in 1:2, a in 1:3
        Λa = Λoneν[layer, a, :, :]
        B[layer, a] = prefactor * quadratic_drive(F, Dν * Λa * Dν)
    end

    @views for a in 1:3, b in 1:3
        Λa = Λoneν[l, a, :, :]
        Λb = Λoneν[lp, b, :, :]
        RIterm = Dν * Λa * Dν * Λb * Dν + Dν * Λb * Dν * Λa * Dν
        RI[a, b] = prefactor * quadratic_drive(F, RIterm)

        Λab = Λfullν[a, b, :, :]
        RII[a, b] = prefactor * quadratic_drive(F, Dν * Λab * Dν)
    end

    return B, RI, RII, RI .+ RII
end

function integrate_frequency_B(Bν, Ωs)
    @assert size(Bν, 3) == length(Ωs)
    B = zeros(ComplexF64, size(Bν, 1), size(Bν, 2))
    for iν in 2:length(Ωs)
        dν = Ωs[iν] - Ωs[iν - 1]
        B .+= 0.5 * dν .* (Bν[:, :, iν] .+ Bν[:, :, iν - 1]) ./ (2π)
    end
    return B
end

function integrate_frequency_R(Rν, Ωs)
    @assert size(Rν, 3) == length(Ωs)
    R = zeros(ComplexF64, size(Rν, 1), size(Rν, 2))
    for iν in 2:length(Ωs)
        dν = Ωs[iν] - Ωs[iν - 1]
        R .+= 0.5 * dν .* (Rν[:, :, iν] .+ Rν[:, :, iν - 1]) ./ (2π)
    end
    return R
end

function monochromatic_static_light_terms_band(components, ks, θ; layer_pair=(1, 2), Vval=V,
                                               ηD=ηΩ, ηΛ=ηΩ)
    B = zeros(ComplexF64, 2, 3)
    RI = zeros(ComplexF64, 3, 3)
    RII = zeros(ComplexF64, 3, 3)
    samples = Any[]

    for comp in components
        ν = comp.frequency
        Dν_all, _, _ = precompute_exciton_D([ν], ks, θ; V=Vval, η=ηD)
        Λoneν_all, _ = lambda_one_spin_layers_band([ν], ks, θ; η=ηΛ)
        Λfullν_all, _ = lambda_layerpair_full_band([ν], ks, θ; layer_pair=layer_pair, η=ηΛ)
        Dν = Dν_all[:, :, 1]
        Λoneν = Λoneν_all[:, :, :, :, 1]
        Λfullν = Λfullν_all[:, :, :, :, 1]

        Bc, RIc, RIIc, Rtotc = contract_static_light_at_frequency(
            Dν, Λoneν, Λfullν, comp.vector; layer_pair=layer_pair)
        B .+= Bc
        RI .+= RIc
        RII .+= RIIc
        push!(samples, (; label=comp.label, frequency=ν, vector=comp.vector,
                        B=Bc, RI=RIc, RII=RIIc, Rtot=Rtotc))
    end

    return B, RI, RII, RI .+ RII, samples
end

function compute_light_saddle_terms_band(Ωs, ks, θ; layer_pair=(1, 2), Vval=V,
                                         ηD=ηΩ, ηΛ=ηΩ, FΩs=nothing,
                                         drive_kwargs...)
    D, Π, dataD = precompute_exciton_D(Ωs, ks, θ; V=Vval, η=ηD)
    Λone, dataΛone = lambda_one_spin_layers_band(Ωs, ks, θ; η=ηΛ)
    Λfull, dataΛfull = lambda_layerpair_full_band(Ωs, ks, θ; layer_pair=layer_pair, η=ηΛ)

    if FΩs === nothing
        FΩs_local, drive_info = make_drive_spectrum_with_info(Ωs; drive_kwargs...)
    else
        FΩs_local = FΩs
        drive_info = (; kind=:external_array, use_frequency_integral=true)
    end

    if hasproperty(drive_info, :use_frequency_integral) && !drive_info.use_frequency_integral
        B = zeros(ComplexF64, 2, 3)
        RI = zeros(ComplexF64, 3, 3)
        RII = zeros(ComplexF64, 3, 3)
        Bν = zeros(ComplexF64, 2, 3, length(Ωs))
        RIν = zeros(ComplexF64, 3, 3, length(Ωs))
        RIIν = zeros(ComplexF64, 3, 3, length(Ωs))

        B, RI, RII, Rtot, drive_samples = monochromatic_static_light_terms_band(
            drive_info.components, ks, θ; layer_pair=layer_pair, Vval=Vval, ηD=ηD, ηΛ=ηΛ)

        for sample in drive_samples
            idx = nearest_frequency_index(Ωs, sample.frequency)
            Bν[:, :, idx] .+= sample.B
            RIν[:, :, idx] .+= sample.RI
            RIIν[:, :, idx] .+= sample.RII
        end
        Rtotν = RIν .+ RIIν
    else
        Bν = light_effective_field_integrand_from_drive(D, Λone, FΩs_local)
        RIν = light_kernel_I_integrand_from_drive(D, Λone, FΩs_local; layer_pair=layer_pair)
        RIIν = light_kernel_II_integrand_from_drive(D, Λfull, FΩs_local)
        Rtotν = RIν .+ RIIν

        B = integrate_frequency_B(Bν, Ωs)
        RI = integrate_frequency_R(RIν, Ωs)
        RII = integrate_frequency_R(RIIν, Ωs)
        Rtot = RI .+ RII
        drive_samples = Any[]
    end

    return (; D, Π, dataD, Λone, dataΛone, Λfull, dataΛfull, FΩs=FΩs_local,
            drive_info, drive_samples, Bν, RIν, RIIν, Rtotν,
            B, RI, RII, Rtot, layer_pair, θ, ηD, ηΛ)
end;

## Fixed-angle static benchmark

This is the first production cell. It computes the $\Omega=0$ benchmark for one magnetic angle and one layer pair. In this limit the multifrequency vertices reduce to the one-frequency Lehmann vertices already used in the previous notebooks.

For a monochromatic drive the code applies the source frequency analytically by evaluating the static contractions at $\nu=\omega_L$ for complex light, or at $\nu=\pm\omega_L$ for real harmonic light. For a Gaussian pulse it performs the remaining $d\nu/(2\pi)$ spectral average.

In [ ]:
# -----------------------------
# Editable parameters for the light-induced Ω=0 benchmark
# -----------------------------
light_layer_pair = (1, 2)
light_drive_channel = 1          # 1=0, 2=x, 3=y, 4=z in the exciton-channel basis
light_drive_envelope = :monochromatic_real # :monochromatic_real, :monochromatic_complex, :gaussian, or :flat_debug
light_drive_amplitude = 1.0
light_drive_phase = 0.0
ωL_light = 5.0
σt_light = 7.0
t0_light = 30.0
normalize_drive_peak = true

ηD_light = ηΩ
ηΛ_light = ηΩ

println("Computing light-induced Ω=0 saddle benchmark...")
@time light_terms = compute_light_saddle_terms_band(
    Ωs, ks, θ;
    layer_pair=light_layer_pair,
    Vval=V,
    ηD=ηD_light,
    ηΛ=ηΛ_light,
    envelope=light_drive_envelope,
    channel=light_drive_channel,
    amplitude=light_drive_amplitude,
    phase=light_drive_phase,
    ωL=ωL_light,
    σt=σt_light,
    t0=t0_light,
    normalize_peak=normalize_drive_peak,
)

println("D size: ", size(light_terms.D))
println("Lambda one size: ", size(light_terms.Λone))
println("Lambda two-derivative size: ", size(light_terms.Λfull))
println("stored frequency samples: RI ", size(light_terms.RIν), ", RII ", size(light_terms.RIIν))
println("finite stored samples: ", all(isfinite, light_terms.Rtotν))
println("finite static kernel: ", all(isfinite, light_terms.Rtot))
println("drive kind: ", light_terms.drive_info.kind)
if hasproperty(light_terms.drive_info, :components)
    for comp in light_terms.drive_info.components
        coeff = comp.vector[light_drive_channel]
        println("  ", comp.label, ": analytic frequency ν=", comp.frequency,
                ", coefficient=", coeff)
    end
end
println("Integrated/evaluated Ω=0 total tensor Re Rtot:")
show(stdout, "text/plain", real.(light_terms.Rtot)); println()

In [ ]:
# -----------------------------
# Plot helpers
# -----------------------------
function project_light_value(z, part)
    part === abs && return abs(z)
    return part(z)
end

function clean_light_line(y; tol=1e-10)
    out = collect(y)
    if maximum(abs.(out)) < tol
        out .= 0.0
    else
        out = [abs(v) < tol ? 0.0 : v for v in out]
    end
    return out
end

function light_part_label(part)
    part === real && return "Re"
    part === imag && return "Im"
    part === abs && return "Abs"
    return string(part)
end;


## Figure 1: source and exciton propagator

This figure checks whether the chosen light frequency overlaps the exciton resonance. For monochromatic light, the left panel only marks the analytic frequencies $\nu=\pm\omega_L$ or $\nu=+\omega_L$; it is not a numerical delta representation. The right panel is the diagonal exciton spectral weight $-2\operatorname{Im}D^R_{\mu\mu}(\nu)$.

In [ ]:
# -----------------------------
# Figure 1: drive spectrum and exciton propagator
# -----------------------------
Fweight = abs2.(light_terms.FΩs[light_drive_channel, :])
Ddiag_spectral = [-2 * imag(light_terms.D[μ, μ, iΩ]) for μ in 1:4, iΩ in axes(light_terms.D, 3)]

fig, axs = subplots(1, 2, figsize=(11.0, 3.8), sharex=true)
if light_terms.drive_info.kind in (:monochromatic_real, :monochromatic_complex)
    nz = findall(x -> x > 1e-12 * maximum(Fweight), Fweight)
    axs[1].vlines(Ωs[nz], zeros(length(nz)), Fweight[nz], color="#111111", lw=1.8)
    axs[1].plot(Ωs[nz], Fweight[nz], "o", color="#111111", ms=4)
else
    axs[1].plot(Ωs, Fweight, lw=1.8, color="#111111")
end
axs[1].set_title("Drive weight in channel " * channel_labels[light_drive_channel])
axs[1].set_xlabel("omega")
axs[1].set_ylabel("|F_mu(omega)|^2")
axs[1].grid(alpha=0.18)

for μ in 1:4
    axs[2].plot(Ωs, Ddiag_spectral[μ, :], lw=1.35, label="D_" * channel_labels[μ] * channel_labels[μ])
end
axs[2].set_title("Exciton propagator")
axs[2].set_xlabel("omega")
axs[2].set_ylabel("-2 Im D^R")
axs[2].legend(frameon=false, fontsize=8)
axs[2].grid(alpha=0.18)
fig.suptitle("Light source and BSE propagator, theta=" * string(θ) * ", V=" * string(V))
fig.tight_layout(rect=(0, 0, 1, 0.92))

## Figure 2: first derivative field, $\Omega=0$ benchmark

This figure shows the stored source-frequency samples of the static field contribution. For a Gaussian pulse these samples are the integrand

$$\mathcal{B}_{l,a}(\nu)=-F^\dagger(\nu)D^R(\nu)\Lambda_l^{a,R}(\nu)D^R(\nu)F(\nu),$$

which is integrated over $d\nu/(2\pi)$. For a monochromatic source, the plotted nonzero points are only markers for the analytic evaluations at $\nu=\pm\omega_L$ or $\nu=\omega_L$.

In [ ]:
# -----------------------------
# Figure 2: first derivative, coherent light-induced field integrand
# -----------------------------
field_layer = 1
field_part = real  # real, imag, or abs
field_label = light_part_label(field_part)
field_zero_tol = 1e-10
field_xlim = (first(Ωs), last(Ωs))

fig, ax = subplots(1, 1, figsize=(7.2, 4.0))
for a in 1:3
    y = clean_light_line([project_light_value(light_terms.Bν[field_layer, a, iν], field_part)
                          for iν in axes(light_terms.Bν, 3)]; tol=field_zero_tol)
    ax.plot(Ωs, y, lw=1.55, label="B_" * spin_labels[a])
end
ax.axhline(0.0, color="0.35", lw=0.7, alpha=0.5)
ax.set_xlabel("omega")
ax.set_ylabel(field_label * " B_light integrand")
ax.set_xlim(field_xlim...)
ax.set_title("Coherent saddle field integrand on layer " * string(field_layer) *
             "; integrated Re B = " * string(round.(real.(light_terms.B[field_layer, :]); sigdigits=3)))
ax.grid(alpha=0.18)
ax.legend(frameon=false, fontsize=9)
fig.tight_layout()


## Figure 3: $R^{I,\mathrm{light}}$ at $\Omega=0$

This panel shows the zero-response-frequency limit of the two-one-vertex term,

$$\mathcal{R}^{I}_{ll',ab}(\nu;\Omega=0)
=-F^\dagger(\nu)\left[
D^R(\nu)\Lambda_l^{a,R}(\nu)D^R(\nu)\Lambda_{l'}^{b,R}(\nu)D^R(\nu)
+D^R(\nu)\Lambda_{l'}^{b,R}(\nu)D^R(\nu)\Lambda_l^{a,R}(\nu)D^R(\nu)
\right]F(\nu).$$

For a Gaussian pulse this is integrated over $\nu$. For a monochromatic source the nonzero plotted points mark the analytic source-frequency evaluations.

In [ ]:
# -----------------------------
# Spectral integrand: RI
# -----------------------------
light_tensor_part = real  # real, imag, or abs
light_tensor_label = light_part_label(light_tensor_part)
light_tensor_zero_tol = 1e-10
light_tensor_xlim = (first(Ωs), last(Ωs))
Rplot = light_terms.RIν

fig, axs = subplots(3, 3, figsize=(10.4, 8.0), sharex=true)
for a in 1:3, b in 1:3
    ax = axs[a, b]
    y = clean_light_line([project_light_value(Rplot[a, b, iν], light_tensor_part)
                          for iν in axes(Rplot, 3)]; tol=light_tensor_zero_tol)
    ax.plot(Ωs, y, lw=1.35, color=(a == b ? "#1f77b4" : "#d62728"))
    ax.axhline(0.0, color="0.35", lw=0.6, alpha=0.45)
    ax.set_title("RI_" * spin_labels[a] * "," * spin_labels[b])
    ax.grid(alpha=0.16)
    ax.set_xlim(light_tensor_xlim...)
    if a == 3
        ax.set_xlabel("omega")
    end
    if b == 1
        ax.set_ylabel(light_tensor_label * " RI(nu) sample")
    end
end
fig.suptitle("Light-induced RI integrand: D Lambda_a D Lambda_b D + reverse, layers " * string(light_layer_pair) *
             ", drive " * channel_labels[light_drive_channel])
fig.tight_layout(rect=(0, 0, 1, 0.94))


## Figure 4: $R^{II,\mathrm{light}}$ at $\Omega=0$

This panel shows the zero-response-frequency limit of the two-magnetization-vertex term,

$$\mathcal{R}^{II}_{ll',ab}(\nu;\Omega=0)
=-F^\dagger(\nu)D^R(\nu)\Lambda_{ll'}^{ab,R}(\nu)D^R(\nu)F(\nu).$$

The full $R^{II}(\Omega)$ would require the multifrequency object $\Lambda_{ll'}^{ab,R}(\nu,\nu;\Omega)$.

In [ ]:
# -----------------------------
# Spectral integrand: RII
# -----------------------------
light_tensor_part = real  # real, imag, or abs
light_tensor_label = light_part_label(light_tensor_part)
light_tensor_zero_tol = 1e-10
light_tensor_xlim = (first(Ωs), last(Ωs))
Rplot = light_terms.RIIν

fig, axs = subplots(3, 3, figsize=(10.4, 8.0), sharex=true)
for a in 1:3, b in 1:3
    ax = axs[a, b]
    y = clean_light_line([project_light_value(Rplot[a, b, iν], light_tensor_part)
                          for iν in axes(Rplot, 3)]; tol=light_tensor_zero_tol)
    ax.plot(Ωs, y, lw=1.35, color=(a == b ? "#1f77b4" : "#d62728"))
    ax.axhline(0.0, color="0.35", lw=0.6, alpha=0.45)
    ax.set_title("RII_" * spin_labels[a] * "," * spin_labels[b])
    ax.grid(alpha=0.16)
    ax.set_xlim(light_tensor_xlim...)
    if a == 3
        ax.set_xlabel("omega")
    end
    if b == 1
        ax.set_ylabel(light_tensor_label * " RII(nu) sample")
    end
end
fig.suptitle("Light-induced RII integrand: D Lambda_ab D, layers " * string(light_layer_pair) *
             ", drive " * channel_labels[light_drive_channel])
fig.tight_layout(rect=(0, 0, 1, 0.94))


## Figure 5: total $\Omega=0$ light-induced curvature

The static benchmark plotted here is

$$\mathcal{R}^{\mathrm{light}}(\nu;\Omega=0)
=\mathcal{R}^{I,\mathrm{light}}(\nu;\Omega=0)
+\mathcal{R}^{II,\mathrm{light}}(\nu;\Omega=0).$$

This should not be read as the full dynamic $R(\Omega)$; it is the $\Omega=0$ limit used to check signs, channels, and relative sizes before implementing the multifrequency vertices.

In [ ]:
# -----------------------------
# Spectral integrand: Rtot
# -----------------------------
light_tensor_part = real  # real, imag, or abs
light_tensor_label = light_part_label(light_tensor_part)
light_tensor_zero_tol = 1e-10
light_tensor_xlim = (first(Ωs), last(Ωs))
Rplot = light_terms.Rtotν

fig, axs = subplots(3, 3, figsize=(10.4, 8.0), sharex=true)
for a in 1:3, b in 1:3
    ax = axs[a, b]
    y = clean_light_line([project_light_value(Rplot[a, b, iν], light_tensor_part)
                          for iν in axes(Rplot, 3)]; tol=light_tensor_zero_tol)
    ax.plot(Ωs, y, lw=1.35, color=(a == b ? "#1f77b4" : "#d62728"))
    ax.axhline(0.0, color="0.35", lw=0.6, alpha=0.45)
    ax.set_title("Rtot_" * spin_labels[a] * "," * spin_labels[b])
    ax.grid(alpha=0.16)
    ax.set_xlim(light_tensor_xlim...)
    if a == 3
        ax.set_xlabel("omega")
    end
    if b == 1
        ax.set_ylabel(light_tensor_label * " Rtot(nu) sample")
    end
end
fig.suptitle("Total coherent light-induced curvature integrand RI + RII, layers " * string(light_layer_pair) *
             ", drive " * channel_labels[light_drive_channel])
fig.tight_layout(rect=(0, 0, 1, 0.94))


## Figure 6: term-by-term comparison

This figure overlays the $\Omega=0$ contributions $R^I$, $R^{II}$, and their sum for selected components. It is a diagnostic of the static benchmark, not yet the full frequency-dependent response kernel.

In [ ]:
# -----------------------------
# Figure 6: selected components, term-by-term spectral-integrand comparison
# -----------------------------
compare_components = [(1, 1), (1, 2), (3, 1), (3, 3)]
compare_part = real  # real, imag, or abs
compare_label = light_part_label(compare_part)
compare_xlim = (first(Ωs), last(Ωs))

fig, axs = subplots(1, length(compare_components), figsize=(3.4 * length(compare_components), 3.6), sharex=true)
for (icol, (a, b)) in enumerate(compare_components)
    ax = axs[icol]
    yI = clean_light_line([project_light_value(light_terms.RIν[a, b, iν], compare_part) for iν in axes(light_terms.RIν, 3)])
    yII = clean_light_line([project_light_value(light_terms.RIIν[a, b, iν], compare_part) for iν in axes(light_terms.RIIν, 3)])
    yT = clean_light_line([project_light_value(light_terms.Rtotν[a, b, iν], compare_part) for iν in axes(light_terms.Rtotν, 3)])
    ax.plot(Ωs, yI, lw=1.25, label="RI")
    ax.plot(Ωs, yII, lw=1.25, label="RII")
    ax.plot(Ωs, yT, lw=1.7, color="black", label="sum")
    ax.axhline(0.0, color="0.35", lw=0.6, alpha=0.45)
    ax.set_title(spin_labels[a] * "," * spin_labels[b])
    ax.set_xlabel("omega")
    ax.set_xlim(compare_xlim...)
    ax.grid(alpha=0.16)
    if icol == 1
        ax.set_ylabel(compare_label * " spectral integrand")
    end
end
axs[1].legend(frameon=false, fontsize=8)
fig.suptitle("Term-by-term comparison for selected spin components")
fig.tight_layout(rect=(0, 0, 1, 0.90))


## Figure 7: integrated static tensors

The previous figures show the frequency-resolved static benchmark. This figure shows the actual $\Omega=0$ tensors after applying the source. In the monochromatic modes the source is applied analytically at the selected light frequencies; in the Gaussian mode the result is a finite-pulse spectral average.

In [ ]:
# -----------------------------
# Figure 7: integrated static light-induced tensors
# -----------------------------
integrated_part = real  # real, imag, or abs
integrated_label = light_part_label(integrated_part)

integrated_tensors = [light_terms.RI, light_terms.RII, light_terms.Rtot]
integrated_titles = ["RI integrated", "RII integrated", "RI + RII integrated"]

fig, axs = subplots(1, 3, figsize=(11.5, 3.6), sharex=true, sharey=true)
vals = [project_light_value(z, integrated_part) for T in integrated_tensors for z in T]
vmax = maximum(abs.(vals))
vmax = vmax > 0 ? vmax : 1.0

for ip in 1:3
    mat = [project_light_value(integrated_tensors[ip][a, b], integrated_part) for a in 1:3, b in 1:3]
    im = axs[ip].imshow(mat, origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    axs[ip].set_title(integrated_titles[ip])
    axs[ip].set_xticks(0:2)
    axs[ip].set_yticks(0:2)
    axs[ip].set_xticklabels(spin_labels)
    axs[ip].set_yticklabels(spin_labels)
    axs[ip].set_xlabel("b")
    if ip == 1
        axs[ip].set_ylabel("a")
    end
    for a in 1:3, b in 1:3
        axs[ip].text(b - 1, a - 1, @sprintf("%.2e", mat[a, b]),
                     ha="center", va="center", fontsize=8, color="black")
    end
end
cbar = fig.colorbar(im, ax=axs, shrink=0.82)
cbar.set_label(integrated_label * " integrated tensor")
fig.suptitle("Static light-induced tensors, layers " * string(light_layer_pair) *
             ", drive " * channel_labels[light_drive_channel])
fig.tight_layout(rect=(0, 0, 1, 0.90))


## Scope of this notebook

The analytic light-induced kernel depends on the response frequency $\Omega$ conjugate to $z-z'$ and on the light/exciton frequency $\nu$ selected by the source. The current numerical implementation evaluates only the $\Omega=0$ static benchmark.

The next nontrivial step is to implement the multifrequency vertices

$$\Lambda_l^{a,R}(\nu+\Omega,\nu),\qquad
\Lambda_{ll'}^{ab,R}(\nu,\nu;\Omega),$$

and then use the analytic contractions above to compute the full $R^{I}(\Omega)$ and $R^{II}(\Omega)$.